In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [23]:
df = pd.read_csv("dataset/train.csv")
tf = pd.read_csv("dataset/test.csv")
df.head()

,sample_id,source_id,has_band_A_spectrum,has_band_B_spectrum,sampling_strategy,sampling_depth_cm,geo_zone_macro,geo_zone_micro,geo_zone_meso,land_cover_type,...,spectral_band_B_PC_6,spectral_band_B_PC_7,spectral_band_B_PC_8,spectral_band_B_PC_9,spectral_band_B_PC_10,spectral_band_B_PC_11,spectral_band_B_PC_12,spectral_band_B_PC_13,spectral_band_B_PC_14,spectral_band_B_PC_15
0,train_00001,Source_01,YES,NO,Auger,0-20,SE,Unknown,State_01,Seasonal Semideciduous Forest,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,train_00002,Source_10,YES,NO,Auger,0-20,MW,Loc_011,State_10,Savannah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,train_00003,Source_04,YES,NO,Auger,0-20,S,Loc_001,State_06,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,train_00004,Source_02,YES,NO,Auger,0-20,N,Unknown,State_07,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,train_00005,Source_04,YES,NO,Auger,0-20,S,Loc_001,State_06,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
dfmiss = df.isnull().sum()
tfmiss = tf.isnull().sum()
print(dfmiss[dfmiss > 0])
print(len(dfmiss[dfmiss > 0]))
print(len(df['biome']))
print(tfmiss[tfmiss > 0])
print(len(tfmiss[tfmiss > 0]))
print(len(tf['biome']))

property_particle_coarse      752
property_particle_fine        767
property_acidity_index       8405
cation_Ca                    1229
cation_Mg                    1230
cation_Na                   10808
cation_exchange_capacity     1229
latitude                     8820
longitude                    8820
spectral_band_B_PC_1         9516
spectral_band_B_PC_2         9516
spectral_band_B_PC_3         9516
spectral_band_B_PC_4         9516
spectral_band_B_PC_5         9516
spectral_band_B_PC_6         9516
spectral_band_B_PC_7         9516
spectral_band_B_PC_8         9516
spectral_band_B_PC_9         9516
spectral_band_B_PC_10        9516
spectral_band_B_PC_11        9516
spectral_band_B_PC_12        9516
spectral_band_B_PC_13        9516
spectral_band_B_PC_14        9516
spectral_band_B_PC_15        9516
dtype: int64
24
11210
property_particle_coarse     178
property_particle_fine       183
property_acidity_index      1969
cation_Ca                    295
cation_Mg                    2

# 1st try (polosan, belum dipisah berdasarkan Band B)

In [25]:
# pake model lightgbm
import lightgbm as lgb

# cation_Na 96% kosong, jadi utk sementara didrop dulu bersamaan sample_id
df_polos = df
df = df.drop(columns=['cation_Na', 'sample_id', 'longitude', 'latitude'])

df['has_band_A_spectrum'] = df['has_band_A_spectrum'].map({'YES': 1, 'NO': 0})
df['has_band_B_spectrum'] = df['has_band_B_spectrum'].map({'YES': 1, 'NO': 0})

target_col = 'property_organic_content'

categorical_cols = ['sampling_depth_cm', 'geo_zone_macro', 'geo_zone_micro', 'geo_zone_meso', 'land_cover_type', 'biome', 'parent_rock_type', 'source_id', 'sampling_strategy']

for col in categorical_cols:
    df[col] = df[col].astype('category')
    tf[col] = tf[col].astype('category')

X = df.drop(columns=[target_col])
y = df[target_col]
X_test = tf.drop(columns=[c for c in ['property_organic_content','cation_Na', 'sample_id', 'longitude', 'latitude'] if c in tf.columns])
y_log  = np.log1p(y)

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmse = []
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'verbose': -1,
    'random_state': 42
}
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
  X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
  train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
  val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=train_data)

  model = lgb.train(
      params,
      train_data,
      num_boost_round=1000,
      valid_sets=[train_data, val_data],
      valid_names=['train', 'valid'],
      callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
      ]
  )

  y_pred = model.predict(X_val, num_iteration=model.best_iteration)

  rmse = np.sqrt(mean_squared_error(y_val, y_pred))
  fold_rmse.append(rmse)
  print(f'Fold {fold+1}  RMSE = {rmse:.4f}')

np.mean(fold_rmse)

Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 8.98613	valid's rmse: 11.3383
[200]	train's rmse: 7.3866	valid's rmse: 11.0318
[300]	train's rmse: 6.42874	valid's rmse: 10.9463
[400]	train's rmse: 5.74093	valid's rmse: 10.8777
[500]	train's rmse: 5.16273	valid's rmse: 10.8417
[600]	train's rmse: 4.67481	valid's rmse: 10.8249
[700]	train's rmse: 4.24662	valid's rmse: 10.8268
Early stopping, best iteration is:
[659]	train's rmse: 4.40262	valid's rmse: 10.814
Fold 1  RMSE = 10.8140
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 9.01282	valid's rmse: 11.4933
[200]	train's rmse: 7.44828	valid's rmse: 11.1724
[300]	train's rmse: 6.48598	valid's rmse: 11.0652
Early stopping, best iteration is:
[330]	train's rmse: 6.2457	valid's rmse: 11.038
Fold 2  RMSE = 11.0380
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 9.01127	valid's rmse: 11.396
[200]	train's rmse: 7.38148	valid's rmse: 11.109
[300]	t

np.float64(11.448394694922195)

rmsenya masih gede (11.408098257415656)

# 2nd try (model dipisah berdasarkan Band B)

In [27]:
band_b_cols = [f'spectral_band_B_PC_{i}' for i in range(1, 16)]

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
  X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

  mask_train_has_B = X_train['spectral_band_B_PC_1'].notna()
  X_train_model1 = X_train[mask_train_has_B]
  y_train_model1 = y_train[mask_train_has_B]

  train_data_1 = lgb.Dataset(X_train_model1, label=y_train_model1, categorical_feature=categorical_cols)
  model_1 = lgb.train(params, train_data_1, num_boost_round=300)

  X_train_model2 = X_train.drop(columns=band_b_cols)

  train_data_2 = lgb.Dataset(X_train_model2, label=y_train, categorical_feature=categorical_cols)
  model_2 = lgb.train(params, train_data_2, num_boost_round=300)

  y_pred = np.zeros(len(X_val))
  mask_val_has_B = X_val['spectral_band_B_PC_1'].notna()
  mask_val_no_B = ~mask_val_has_B

  if mask_val_has_B.sum() > 0:
    y_pred[mask_val_has_B] = model_1.predict(X_val[mask_val_has_B])

  if mask_val_no_B.sum() > 0:
    X_val_model2 = X_val[mask_val_no_B].drop(columns=band_b_cols)
    y_pred[mask_val_no_B] = model_2.predict(X_val_model2)

  rmse = np.sqrt(mean_squared_error(y_val, y_pred))
  fold_rmse.append(rmse)

np.mean(fold_rmse)

np.float64(11.529074191268993)

rmsenya makin gede (11.515403102794732)

In [28]:
# XGBoost doesn't support native categoricals — encode as codes
import xgboost as xgb
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
for col in categorical_cols:
    X_xgb[col]      = X_xgb[col].cat.codes
    X_test_xgb[col] = X_test_xgb[col].cat.codes

xgb_params = {
    'objective':        'reg:squarederror',
    'eval_metric':      'rmse',
    'learning_rate':    0.02,
    'max_depth':        6,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.7,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'tree_method':      'hist',
    'random_state':     42,
    'verbosity':        0,
}

xgb_oof   = np.zeros(len(X))
xgb_test  = np.zeros(len(X_test))
xgb_rmse  = []

for fold, (tr, val) in enumerate(kf.split(X_xgb, y_log)):
    Xtr, Xval = X_xgb.iloc[tr], X_xgb.iloc[val]
    ytr, yval = y_log.iloc[tr], y_log.iloc[val]

    dtr  = xgb.DMatrix(Xtr, label=ytr)
    dval = xgb.DMatrix(Xval, label=yval)

    m = xgb.train(
        xgb_params, dtr,
        num_boost_round=3000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False,
    )

    pred             = np.expm1(m.predict(dval))
    xgb_oof[val]     = pred
    # xgb_test        += np.expm1(m.predict(xgb.DMatrix(X_test_xgb))) / 5

    rmse = np.sqrt(mean_squared_error(y.iloc[val], pred))
    xgb_rmse.append(rmse)
    print(f'Fold {fold+1}  RMSE = {rmse:.4f}')

print(f'\nXGBoost CV RMSE: {np.mean(xgb_rmse):.4f} ± {np.std(xgb_rmse):.4f}')


Fold 1  RMSE = 11.5339
Fold 2  RMSE = 11.0730
Fold 3  RMSE = 11.7307
Fold 4  RMSE = 12.2735
Fold 5  RMSE = 12.9798

XGBoost CV RMSE: 11.9182 ± 0.6558


In [29]:
# Weighted ensemble — try a few weights
for w in [0.5, 0.6, 0.7]:
    ens  = w * lgb_oof + (1 - w) * xgb_oof
    rmse = np.sqrt(mean_squared_error(y, ens))
    print(f'{w:.1f} LGB + {1-w:.1f} XGB  →  OOF RMSE = {rmse:.4f}')

# Final predictions (use best weight, e.g. 0.7 LGB)
W_LGB = 0.7
final_preds = W_LGB * lgb_test + (1 - W_LGB) * xgb_test

sub = test[['sample_id']].copy()
sub['property_organic_content'] = final_preds
sub.to_csv('vtry2.csv', index=False)
print('\nSubmission saved!')
sub.describe()


NameError: name 'lgb_oof' is not defined

# Nambahin feature engineering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mengambil nilai importance berdasarkan 'gain' (seberapa besar kontribusi fitur mengurangi error)
feature_imp = pd.DataFrame(sorted(zip(model.feature_importance(importance_type='gain'), X.columns)), columns=['Value','Feature'])

# Mengurutkan dan mengambil 20 fitur dengan pengaruh paling besar
top_features = feature_imp.sort_values(by="Value", ascending=False).head(20)

# Membuat plot visualisasi
plt.figure(figsize=(10, 8))
sns.barplot(x="Value", y="Feature", data=top_features, hue="Feature", palette="viridis", legend=False)
plt.title('Top 20 Feature Importance - LightGBM (Berdasarkan Gain)')
plt.xlabel('Tingkat Kepentingan Fitur (Gain)')
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.show()